In [1]:
import cobra as cb
import pandas as pd
import numpy as np

model = cb.io.load_json_model("../Cobra_Models/GPR_model_1a.json")

n_exs = len(model.exchanges)
n_genes = len(model.genes)

n_genes_ko = 6
n_envs = 100

genes_ko = np.random.choice(model.genes.list_attr("id"), size=(n_genes_ko), replace=False)
# convert to 1 hot encoding
genes_ko_1hot = np.zeros((n_genes_ko, n_genes))
for i, gene in enumerate(genes_ko):
    gene_index = model.genes.list_attr("id").index(gene)
    genes_ko_1hot[i, gene_index] = 1

envs_sim = np.random.random((n_envs, n_exs-1)) # minus 1 for bio

envs_sim[envs_sim < 0.5] = 0
envs_sim *= -100 
# replace non unique rows with unique rows
envs_sim = np.unique(envs_sim, axis=0)
while envs_sim.shape[0] < n_envs:
    envs_sim_new = np.random.random((n_envs, n_exs-1)) # minus 1 for bio
    envs_sim_new[envs_sim_new < 0.5] = 0
    envs_sim_new *= -100 
    envs_sim = np.vstack((envs_sim, envs_sim_new))
    envs_sim = np.unique(envs_sim, axis=0)

ex_ids = model.exchanges.list_attr("id")
ex_ids.remove("EX_Bio(e)")

# store simulation results
# columns = n_genes + n_exs-1 (minus 1 for bio) +1 (final column is the objective value)
df = pd.DataFrame(columns=[f"{gene}?" for gene in model.genes.list_attr("id")] + [f"{ex}" for ex in ex_ids] + ["Objective"])

for i in range(n_envs):
    for j in range(n_genes_ko):
        with model as mdl_tmp:
            # reset model bounds
            for ex in mdl_tmp.exchanges:
                ex.lower_bound = 0
                ex.upper_bound = 1000

            
                

            # set media
            for k in range(n_exs-1):
                mdl_tmp.exchanges.get_by_id(ex_ids[k]).lower_bound = envs_sim[i, k]

            # knockout gene
            gene_ko = genes_ko[j]
            mdl_tmp.genes.get_by_id(gene_ko).knock_out()

            # simulate
            solution = mdl_tmp.optimize()

            # add row to dataframe
            df.loc[i*n_genes_ko + j] = [0]*(n_genes) + list(envs_sim[i]) + [solution.objective_value]

            # store results
            df.iloc[i*n_genes_ko + j, -1] = solution.objective_value

            # add KO and media info to dataframe
            df.iloc[i*n_genes_ko + j, :n_genes] = genes_ko_1hot[j]
            df.iloc[i*n_genes_ko + j, n_genes:-1] = envs_sim[i]
        
         


print("KO genes:", genes_ko)
print("KO genes 1 hot:")
print(genes_ko_1hot)
print("Simulated environments:")
print(envs_sim)

print("Simulation results:")
print(df)

df.to_csv("simulated_data_mdl_1a.csv", index=False)



Set parameter Username
Set parameter LicenseID to value 2773321
Academic license - for non-commercial use only - expires 2027-02-01
KO genes: ['G3' 'G4' 'G1' 'G5' 'G6' 'G2']
KO genes 1 hot:
[[0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0.]
 [1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 1.]
 [0. 1. 0. 0. 0. 0.]]
Simulated environments:
[[-99.23673382  -0.        ]
 [-99.10906794  -0.        ]
 [-98.4387936  -75.80117884]
 [-98.2972681  -68.71953706]
 [-97.78685445  -0.        ]
 [-97.75753184 -72.06093551]
 [-97.67046164 -77.58943888]
 [-97.54974153 -62.63353261]
 [-97.54112804  -0.        ]
 [-97.46675595 -90.06387656]
 [-93.17339994 -71.14414758]
 [-92.17151227  -0.        ]
 [-91.95464228 -54.67791991]
 [-90.49326791  -0.        ]
 [-89.5506895   -0.        ]
 [-89.51748756 -85.05528437]
 [-89.49749487 -75.43229898]
 [-89.03043807  -0.        ]
 [-87.68556894  -0.        ]
 [-87.24813338 -96.63304743]
 [-87.23608129  -0.        ]
 [-87.12097673  -0.        ]
 [-85.42695126 -71.35